In [ ]:
# התקנת הספרייה
!pip install -q yt-dlp

import yt_dlp
import zipfile
import os
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML, FileLink

# --- רכיבי התקדמות ---
progress_bar = widgets.FloatProgress(
    value=0.0,
    min=0.0,
    max=100.0,
    description='התקדמות:',
    bar_style='info',
    style={'bar_color': '#59a1ff'},
    layout={'width': '95%'}
)

status_label = widgets.Label(value="ממתין להפעלה...")

# --- פונקציה ליצירת כפתור "בכל מחיר" ---
def show_big_download_button(filename):
    btn_html = f"""
    <div style="text-align: right; direction: rtl; margin-top: 10px;">
        <h3 style="color: green;">✅ הקובץ מוכן!</h3>
        <a href="{filename}" target="_blank" download>
            <button style="
                background-color: #28a745; 
                color: white; 
                font-size: 20px; 
                padding: 15px 40px; 
                border: none; 
                border-radius: 10px; 
                cursor: pointer; 
                font-weight: bold; 
                box-shadow: 0 4px 8px rgba(0,0,0,0.2);
                width: 100%;
            ">
                📥 לחץ כאן להורדת: {filename}
            </button>
        </a>
        <p style="font-size: 12px; color: #666; margin-top: 5px;">
            (אם הלחיצה לא עובדת, הקובץ נמצא גם בתפריט בצד ימין תחת תיקיית Output)
        </p>
    </div>
    """
    return HTML(btn_html)

def progress_hook(d):
    if d['status'] == 'downloading':
        try:
            p = d.get('_percent_str', '0%').replace('%', '')
            progress_bar.value = float(p)
            status_label.value = f"📥 מוריד: {d['_percent_str']} | מהירות: {d.get('_speed_str', 'N/A')}"
        except:
            pass
    elif d['status'] == 'finished':
        progress_bar.value = 100
        progress_bar.bar_style = 'success'
        status_label.value = "✅ הושלם! מעבד..."

# פונקציית ההורדה והדחיסה
def start_process(button):
    raw_url = url_input.value.strip()
    quality = quality_dropdown.value
    zip_name = zip_input.value.strip()
    
    # איפוס
    progress_bar.value = 0
    progress_bar.bar_style = 'info'
    status_label.value = "מתחיל..."
    
    if not raw_url:
        with output:
            clear_output()
            print("❌ שגיאה: נא להזין קישור לוידאו.")
        return

    urls = []
    if multi_download_checkbox.value:
        urls = [u.strip() for u in raw_url.replace('\n', ',').split(',') if u.strip()]
    else:
        urls = [raw_url]

    if not zip_name and not use_title_checkbox.value:
        zip_name = "my_download"
    if not zip_name.endswith('.zip'):
        zip_name += '.zip'
    
    # --- כאן מוצג הטקסט בהתחלה ---
    with output:
        clear_output()
        display(status_label, progress_bar)
        print(f"⏳ מתחיל בהורדה של {len(urls)} קישורים...")

    output_dir = 'temp_download_folder'
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
    
    # הגדרת הפורמט
    format_sel = 'bestvideo+bestaudio/best'
    post_processors = []

    if quality == '1080p':
        format_sel = 'bestvideo[height<=1080]+bestaudio/best'
    elif quality == '720p':
        format_sel = 'bestvideo[height<=720]+bestaudio/best'
    elif quality == '480p':
        format_sel = 'bestvideo[height<=480]+bestaudio/best'
    elif quality == 'Audio Only (MP3)':
        format_sel = 'bestaudio/best'
        # המרה ל-MP3
        post_processors.append({
            'key': 'FFmpegExtractAudio',
            'preferredcodec': 'mp3',
            'preferredquality': '192',
        })

    ydl_opts = {
        'outtmpl': f'{output_dir}/%(title)s.%(ext)s',
        'format': format_sel,
        'quiet': True,
        'ignoreerrors': True,
        'progress_hooks': [progress_hook],
        # מבטיח שהקובץ הסופי יהיה MP4 (אם זה וידאו)
        'merge_output_format': 'mp4' if quality != 'Audio Only (MP3)' else None,
        'postprocessors': post_processors,
    }

    try:
        with yt_dlp.YoutubeDL(ydl_opts) as ydl:
            ydl.download(urls)
        
        downloaded_files = os.listdir(output_dir)
        if not downloaded_files:
            print("❌ לא נמצאו קבצים.")
            return

        if use_title_checkbox.value:
            first_file = downloaded_files[0]
            base_name = os.path.splitext(first_file)[0]
            zip_name = "".join([c for c in base_name if c.isalnum() or c in (' ', '-', '_')]).strip() + '.zip'

        status_label.value = "📦 דוחס ל-ZIP..."
        
        with zipfile.ZipFile(zip_name, 'w', zipfile.ZIP_DEFLATED) as zipf:
            for root, dirs, files in os.walk(output_dir):
                for file in files:
                    file_path = os.path.join(root, file)
                    zipf.write(file_path, arcname=file)
                    os.remove(file_path)
        
        if os.path.exists(output_dir):
            os.rmdir(output_dir)
            
        # --- הצגת הכפתור הסופי ---
        with output:
            clear_output()
            display(status_label, progress_bar)
            display(show_big_download_button(zip_name))
            display(FileLink(zip_name))

    except Exception as e:
        status_label.value = "❌ שגיאה"
        with output:
            print(f"❌ אירעה שגיאה: {e}")

# --- עיצוב הממשק ---
style = {'description_width': 'initial'}
url_input = widgets.Textarea(
    description='קישור/ים:',
    placeholder='הדבק כאן קישור...',
    style=style,
    layout={'width': '95%', 'height': '60px'}
)

# עדכון רשימת האפשרויות
quality_dropdown = widgets.Dropdown(
    options=['Best Quality', '1080p', '720p', '480p', 'Audio Only (MP3)'],
    value='Best Quality',
    description='איכות:',
    style=style
)

zip_input = widgets.Text(value='my_videos', description='שם קובץ ה-ZIP:', style=style)
use_title_checkbox = widgets.Checkbox(value=False, description='לקרוא ל-ZIP בשם הוידאו', style=style)
multi_download_checkbox = widgets.Checkbox(value=False, description='הורדה מרובה', style=style)

download_button = widgets.Button(
    description='התחל עיבוד',
    button_style='primary',
    icon='play',
    layout={'width': '50%'}
)

output = widgets.Output()

download_button.on_click(start_process)

display(
    url_input,
    quality_dropdown,
    zip_input,
    use_title_checkbox,
    multi_download_checkbox,
    download_button,
    output
)